# Introduction

This notebook contains an analysis of water samples for various viruses. The focus is on the taxonomy and Baltimore classification of these viruses. This data is compared with weather data. The weather data consists of the average values from the five days preceding the day the sample was collected at each location.

# Imports

In [21]:
# For data import
from pathlib import Path # for handling file paths
import re # for regular expressions to create valid dataframe names

# For data merging
import os # for file and directory operations

import pandas as pd # for data manipulation
import numpy as np # for numerical operations
import seaborn as sns # for data visualization
import matplotlib.pyplot as plt # for plotting
from skbio.diversity.alpha import shannon # for calculating alpha diversity
from skbio.diversity import beta_diversity # for calculating beta diversity
from skbio.stats.ordination import pcoa # for principal coordinates analysis
from scipy.stats import pearsonr # for correlation analysis

# Data Import

## Load Samplings with viruses information

In [22]:
def find_folder(*path_parts):
    possible_folders = [
        Path(*path_parts),
        Path("..") / Path(*path_parts),
    ]

    for folder in possible_folders:
        if folder.exists():
            return folder

    raise FileNotFoundError(f"Folder not found: {'/'.join(path_parts)}")


def load_merged_reads_dataframes(folder_path):
    created_dataframes = {}

    for csv_file in sorted(Path(folder_path).glob("*.csv")):
        dataframe_name = csv_file.stem
        df = pd.read_csv(csv_file)

        created_dataframes[dataframe_name] = df

    return created_dataframes


samplings_dir = find_folder("data", "samplings_with_virus_information")
merged_reads_dataframes = load_merged_reads_dataframes(samplings_dir)

# Control
print("Merged reads DataFrames:", merged_reads_dataframes)

Merged reads DataFrames: {'df_copenhagen_merged_reads_with_virus_information':                 name    taxid  ERR14789322  ERR14789323  ERR14789324  \
0     Mastadenovirus    10509          NaN          NaN          NaN   
1     Mastadenovirus    10509          NaN          NaN          NaN   
2     Mastadenovirus    10509          NaN          NaN          NaN   
3     Mastadenovirus    10509          NaN          NaN          NaN   
4     Mastadenovirus    10509          NaN          NaN          NaN   
...              ...      ...          ...          ...          ...   
1693    Baldwinvirus  3153089          NaN          NaN          NaN   
1694      Hodnevirus  3153200          NaN          NaN          NaN   
1695      Risoevirus  3424972          NaN          NaN          NaN   
1696   Margaeryvirus  3425048          NaN          NaN          NaN   
1697    Nicoomyvirus  3425078          NaN         0.18          NaN   

      ERR14789325  ERR14789326  ERR14789327  ERR14789328

# General Analysis of Measured Viruses

# Completeness of Virus Data

In [23]:
# Check how many lines contain virus information
for dataframe_name, df in merged_reads_dataframes.items():
    has_virus_info = df["taxid"].notna()

    print(
        f"{dataframe_name}: "
        f"{has_virus_info.sum()} Zeilen mit Virusinformationen, "
        f"{(~has_virus_info).sum()} Zeilen ohne Virusinformationen"
    )

df_copenhagen_merged_reads_with_virus_information: 1698 Zeilen mit Virusinformationen, 0 Zeilen ohne Virusinformationen
df_global_merge_with_virus_information: 4366 Zeilen mit Virusinformationen, 0 Zeilen ohne Virusinformationen
df_guangzhou_merged_reads_with_virus_information: 1376 Zeilen mit Virusinformationen, 0 Zeilen ohne Virusinformationen
df_kualalumpur_merged_reads_with_virus_information: 1236 Zeilen mit Virusinformationen, 0 Zeilen ohne Virusinformationen
df_melbourne_merged_reads_with_virus_information: 2880 Zeilen mit Virusinformationen, 0 Zeilen ohne Virusinformationen
df_quito_merged_reads_with_virus_information: 1178 Zeilen mit Virusinformationen, 0 Zeilen ohne Virusinformationen
df_regina_merged_reads_with_virus_information: 1967 Zeilen mit Virusinformationen, 0 Zeilen ohne Virusinformationen
df_seattle_merged_reads_with_virus_information: 1594 Zeilen mit Virusinformationen, 0 Zeilen ohne Virusinformationen
df_yaounde_merged_reads_with_virus_information: 2585 Zeilen mit 

# Compilation of the Dictionary for the Cities

In [24]:
# Locate both data directories independently of the notebook's working directory
samplings_dir = Path(find_folder("data", "samplings_with_virus_information"))
weather_dir = Path(find_folder("data", "samplings"))

# Define the filename suffix for each data type
samplings_suffix = "_merged_reads_with_virus_information.csv"
weather_suffix = "_weather.csv"

# Map normalized city names to their sampling filenames
samplings_files = {
    file.name.removeprefix("df_")
             .removesuffix(samplings_suffix)
             .replace("_", "")
             .lower(): file.name
    for file in sorted(samplings_dir.glob(f"*{samplings_suffix}"))
}

# Map normalized city names to their weather filenames
weather_files = {
    file.name.removesuffix(weather_suffix)
             .replace("_", "")
             .lower(): file.name
    for file in sorted(weather_dir.glob(f"*{weather_suffix}"))
}

# Select cities for which both sampling and weather files exist
common_cities = samplings_files.keys() & weather_files.keys()

# Store the matching sampling and weather filenames as tuples
cities = {
    city: (samplings_files[city], weather_files[city])
    for city in sorted(common_cities)
}

# Report missing matches clearly instead of returning an empty dictionary
if not cities:
    raise ValueError(
        f"No matching file pairs found. "
        f"Sampling directory: {samplings_dir.resolve()}, "
        f"weather directory: {weather_dir.resolve()}"
    )

print(cities)

{'copenhagen': ('df_copenhagen_merged_reads_with_virus_information.csv', 'Copenhagen_weather.csv'), 'guangzhou': ('df_guangzhou_merged_reads_with_virus_information.csv', 'Guangzhou_weather.csv'), 'kualalumpur': ('df_kualalumpur_merged_reads_with_virus_information.csv', 'KualaLumpur_weather.csv'), 'melbourne': ('df_melbourne_merged_reads_with_virus_information.csv', 'Melbourne_weather.csv'), 'quito': ('df_quito_merged_reads_with_virus_information.csv', 'Quito_weather.csv'), 'regina': ('df_regina_merged_reads_with_virus_information.csv', 'Regina_weather.csv'), 'seattle': ('df_seattle_merged_reads_with_virus_information.csv', 'Seattle_weather.csv'), 'yaounde': ('df_yaounde_merged_reads_with_virus_information.csv', 'Yaounde_weather.csv')}


In [25]:
# --- Mapping of the original CSV columns to shorter names in a dictionary ---
weather_features = {
    'temperature_2m_mean (°C)':      'Temperatur',
    'rain_sum (mm)':                 'Regen',
    'relative_humidity_2m_mean (%)': 'Luftfeuchtigkeit',
}

# TODO: Nachprüfen, ob die ENA-Beschreibung korrekt ist
# --- Loading the ENA metadata ---
# It links each sample ID to its sampling date
# The ena_data.tsv file contains information about the samples analyzed
print("Lade ENA-Metadaten...")
ena_data_dir = Path(find_folder('data', 'climate_analysis'))
ena_data = pd.read_csv(ena_data_dir / 'ena_data.tsv', sep='\t')
print(f"ENA-Daten: {len(ena_data)} Samples\n")

# TODO: Nachprüfen, ob die PCoA-Beschreibung korrekt ist
# --- Creation of the Results Dictionaries ---
# For each city, the following data is stored separately:
#   - the adjusted analysis data,
#   - the virus abundance matrix,
#   - the PCoA results are stored separately
total_data = {}
total_samplings  = {}
total_pcoa   = {} # It's the result of a Principal Coordinates Analysis (German: Hauptkoordinatenanalyse)


# --- Processing all cities and their previously assigned file pairs ---
for city, (sampling_file, weather_file) in cities.items():
    # Load the virus table from the sampling directory
    sampling_path = samplings_dir / sampling_file
    sampling_df = pd.read_csv(sampling_path, index_col=0)

    # Keep only sequencing-run columns
    # All other columns contain taxonomy metadata
    sample_columns = sampling_df.columns[
        sampling_df.columns.isin(ena_data['run_accession'])
    ]
    if sample_columns.empty:
        raise ValueError(f'No ENA sample columns found in {sampling_path}')
    sampling_df = sampling_df.loc[:, sample_columns]
    
    # interpretation of missing abundances as 0:
    # Abundance describes how frequently or in what quantity a virus occurs in a sample)
    sampling_df = sampling_df.fillna(0)
    
    # Transpose the DataFrame
    #   - Rows = samples,
    #   - Columns = viruses
    sampling_df = sampling_df.T

    # Calculating alpha diversity:
    # Alpha diversity describes the biological diversity within a single sample
    # The Shannon index describes the diversity within a single sample based on virus abundances
    alpha_div = pd.DataFrame([
        {
            'Sample': sid,
            'Shannon': shannon(sampling_df.loc[sid].values)
        }
        
        for sid in sampling_df.index
    ])

    # TODO: Nachprüfen, ob die run_accession-Beschreibung korrekt ist
    # Filtering ENA entries
    # Select only ENA entries whose `run_accession` appears in the virus data
    # The Run Accession is a unique identifier for a single sequencing run in the European Nucleotide Archive (ENA)
    #
    # Then, only the sample ID and sampling date are retained and renamed
    ena_city = (
        ena_data[ena_data['run_accession'].isin(alpha_div['Sample'])]
        [['run_accession', 'collection_date']]
        .copy()
        .rename(columns={'run_accession': 'Sample', 'collection_date': 'Date'})
    )
    
    # Converting dates to actual pandas date values
    ena_city['Date'] = pd.to_datetime(ena_city['Date'])
    
    # TODO: Nachprüfen, ob die Shannon-Beschreibung korrekt ist
    # Link Shannon values and sampling data using the shared sample ID
    # 
    # Shannon values are measures of diversity within a sample—in
    # (in our case the Shannon values are measures of the diversity of viruses in a sample)
    # 
    # They take into account:
    #   - How many different viruses are present
    #   - How evenly their abundances are distributed
    # 
    # Shannon = 0: Only one virus is present.
    # Low value: Few viruses, or one virus is strongly dominant.
    # High value: Many viruses are present in relatively equal numbers.
    # 
    # There is no general threshold for “high” or “low.”
    # Shannon values should primarily be compared between comparable samples within the same dataset.
    data = alpha_div.merge(ena_city, on='Sample')

    # Loading weather data for the city
    weather_path = weather_dir / weather_file
    weather_df = pd.read_csv(weather_path)
    
    # Converting the time column in the weather data
    weather_df['time'] = pd.to_datetime(weather_df['time'])

    # Create a new, initially empty result column for each weather characteristic
    for csv_column, col_name in weather_features.items():
        data[col_name] = np.nan
        
        # For each sample, determine the time period from the sampling date back to four days prior
        for i, row in data.iterrows():
            # Select only weather data within this five-day window (inclusive)
            start = row['Date'] - pd.Timedelta(days=4)
            mask = (weather_df['time'] >= start) & (weather_df['time'] <= row['Date'])
            
            # Calculate the mean value of the weather variable and store it in the "Sample" row
            data.at[i, col_name] = weather_df.loc[mask, csv_column].mean()

    # Remove samples that are missing a weather value or the Shannon index
    data_clean = data.dropna(subset=list(weather_features.values()) + ['Shannon'])
    
    # TODO: Nachprüfen, ob die Bray-Curtis-Distanz-Beschreibung korrekt ist
    # Calculation of beta diversity between all sample pairs as the Bray-Curtis distance
    # The Bray-Curtis distance describes how different the composition of two samples is.
    # (In our case, it compares the virus abundances of two samples.)
    # 
    # The values typically range from 0 to 1:
    #   0: Both samples have the same viral composition and the same abundances.
    #   close to 0: The samples are very similar.
    #   close to 1: The samples are very different.
    #   1: They have no viruses in common with positive abundance.
    bray_curtis_distances = beta_diversity('braycurtis', sampling_df.values, sampling_df.index)
    
    # Reducing the distance matrix to a few visualizable dimensions using PCoA
    pcoa_res = pcoa(bray_curtis_distances)

    # Save all results under the city name so they can be retrieved later
    total_data[city] = data_clean
    total_samplings[city] = sampling_df
    total_pcoa[city]  = pcoa_res

# Merge the cleaned data from all cities into a single DataFrame
# `ignore_index=True` creates a new, continuous row index for this purpose
total_samples_df = pd.concat(total_data.values(), ignore_index=True)

# Determine global display limits from all cities so that subsequent charts
# use the same axis and color scales and are directly comparable
global_display_limits = {
    'Temperatur':       (total_samples_df['Temperatur'].min() - 2,       total_samples_df['Temperatur'].max() + 2),
    'Regen':            (total_samples_df['Regen'].min() - 2,            total_samples_df['Regen'].max() + 2),
    'Luftfeuchtigkeit': (total_samples_df['Luftfeuchtigkeit'].min() - 2, total_samples_df['Luftfeuchtigkeit'].max() + 2),
    'Shannon':          (total_samples_df['Shannon'].min() - 0.2,        total_samples_df['Shannon'].max() + 0.2),
}

Lade ENA-Metadaten...
ENA-Daten: 678 Samples



/Users/passis./opt/anaconda3/envs/.venv/lib/python3.13/site-packages/skbio/stats/ordination/_principal_coordinate_analysis.py:222: RuntimeWarning: EIGH: since no value for dimensions is specified, PCoA for all dimensions will be computed, which may result in long computation time if the original distance matrix is large.
  warn(
/Users/passis./opt/anaconda3/envs/.venv/lib/python3.13/site-packages/skbio/stats/ordination/_principal_coordinate_analysis.py:222: RuntimeWarning: EIGH: since no value for dimensions is specified, PCoA for all dimensions will be computed, which may result in long computation time if the original distance matrix is large.
  warn(
/Users/passis./opt/anaconda3/envs/.venv/lib/python3.13/site-packages/skbio/stats/ordination/_principal_coordinate_analysis.py:359: RuntimeWarning: The result contains negative eigenvalues that are large in magnitude, which may suggest result inaccuracy. See Notes for details. The negative-most eigenvalue is -0.001640343165583708 whereas

# Analysis

## General Analysis of Hosts in Data

### Global Data

### Cities

## General Analysis of Human Hosts in Data

### Global Data

### Cities

Analysis of Baltimore Classification